# AlphaDent YOLOv8 Segmentation Submission Notebook (V13 — tiled inference)

This notebook is for **inference and submission only**.

The V13 model is trained on **tiles**, so inference must match: each test image is sliced
into overlapping tiles with the **same** geometry used in training (`src/01`), YOLO predicts
on each tile, the predicted polygons are mapped back to **full-image** coordinates, and
overlapping detections are merged with class-wise NMS. The result is written to `submission.csv`.

The official sample submission format is:

```text
id,patient_id,class_id,confidence,poly
```

where:

- `id` is the row-level prediction ID.
- `patient_id` is the image filename without extension.
- `class_id` is the predicted pathology class ID.
- `confidence` is the model confidence score.
- `poly` is the normalized polygon string in YOLO format: `x1 y1 x2 y2 ... xN yN`,
  **in full-image coordinates** (after untiling).

The tiling parameters here (`TILE_SIZE`, `OVERLAP`) must match the values used to train the
model in `src/01`. This notebook also keeps memory-safe inference settings.</cell id="8a414e3f-a2e9-44fa-8dc9-bb4976b710b9">

## 1. Environment Setup

Kaggle submission notebooks often run with Internet disabled.  
If `ultralytics` is not already installed, this cell can install it from an offline wheel dataset.

If you use offline wheels, add your wheel dataset to the notebook input and set `MANUAL_WHEELS_DIR` if auto-detection does not find it.

In [9]:
!pip install -q ultralytics

In [10]:
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

# This can reduce CUDA memory fragmentation.
# It is best to set it before importing torch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Set this only if auto-detection fails.
# Example:
# MANUAL_WHEELS_DIR = Path("/kaggle/input/ultralytics-wheels")
MANUAL_WHEELS_DIR = None

# Keep this False for official offline submission.
# Set to True only when you are testing in an Internet-enabled notebook.
ALLOW_INTERNET_INSTALL = False


def package_available(package_name: str) -> bool:
    """Return True if a Python package is already importable."""
    return importlib.util.find_spec(package_name) is not None


def find_wheel_dirs(root: Path = Path("/kaggle/input")):
    """Find directories that contain .whl files under /kaggle/input."""
    if not root.exists():
        return []

    wheel_files = sorted(root.rglob("*.whl"))
    wheel_dirs = sorted({p.parent for p in wheel_files})

    # Prefer directories that contain ultralytics wheels.
    wheel_dirs = sorted(
        wheel_dirs,
        key=lambda d: any("ultralytics" in p.name.lower() for p in d.glob("*.whl")),
        reverse=True,
    )
    return wheel_dirs


if not package_available("ultralytics"):
    print("ultralytics is not installed.")

    wheel_dirs = []
    if MANUAL_WHEELS_DIR is not None:
        wheel_dirs.append(Path(MANUAL_WHEELS_DIR))
    wheel_dirs.extend(find_wheel_dirs())

    # Remove duplicated directories while preserving order.
    seen = set()
    wheel_dirs = [d for d in wheel_dirs if not (str(d) in seen or seen.add(str(d)))]

    if len(wheel_dirs) > 0:
        print("Installing ultralytics from offline wheel directories:")
        for d in wheel_dirs:
            print(" -", d)

        cmd = [sys.executable, "-m", "pip", "install", "--no-index"]
        for d in wheel_dirs:
            cmd.extend(["--find-links", str(d)])
        cmd.append("ultralytics")

        subprocess.check_call(cmd)
    elif ALLOW_INTERNET_INSTALL:
        print("No offline wheels found. Installing ultralytics from the Internet.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    else:
        raise RuntimeError(
            "ultralytics is not installed and no offline wheel dataset was found. "
            "Add an offline wheel dataset, set MANUAL_WHEELS_DIR, or enable Internet for testing."
        )
else:
    print("ultralytics is already installed.")

ultralytics is already installed.


## 2. Imports and Global Configuration

Update `MANUAL_MODEL_PATH` if auto-detection selects the wrong checkpoint.

This is the **V13** model — trained on tiles. The key settings:

- model family: `yolov8s-seg` (stock head)
- training: tile-based, `TILE_SIZE=640`, `OVERLAP=0.20`
- per-tile inference image size: `IMG_SIZE = TILE_SIZE`

`TILE_SIZE` and `OVERLAP` here **must match** the values used in `src/01`.

For inference, the most important memory-saving setting is `RETINA_MASKS = False`, which
avoids generating high-resolution masks that can cause CUDA out-of-memory errors.</cell id="7cff2878-4db8-4a46-b4af-f22f65b33ddb">

In [ ]:
import gc
import math
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

# ------------------------------------------------------------
# Model path
# ------------------------------------------------------------

# Set this to the exact best.pt path if auto-detection is not enough.
# Example:
# MANUAL_MODEL_PATH = Path("/kaggle/input/my-alphadent-v13-tile-model/best.pt")
MANUAL_MODEL_PATH = None

# If multiple .pt files are found, this index is used after ranking.
MODEL_INDEX = 0

# Keywords used to rank checkpoint candidates. V13 run names contain "v13_tile".
MODEL_KEYWORDS = ["v13", "tile", "yolov8s", "best"]

# ------------------------------------------------------------
# Tiling parameters  (MUST match src/01 training)
# ------------------------------------------------------------
TILE_SIZE = 640        # tile side in pixels (same as src/01 TILE_SIZE)
OVERLAP = 0.20         # fractional overlap between tiles (same as src/01 OVERLAP)
MERGE_IOU = 0.50       # bbox IoU threshold for merging detections across tiles

# ------------------------------------------------------------
# Inference settings
# ------------------------------------------------------------

# Per-tile YOLO input size. The model was trained at imgsz=TILE_SIZE, so each tile
# is predicted at TILE_SIZE (the tile is fed ~1:1, no extra downscaling).
IMG_SIZE = TILE_SIZE
CONF_THRES = 0.05
IOU_THRES = 0.50       # within-tile NMS during model.predict
MAX_DET = 100

# CUDA OOM prevention
RETINA_MASKS = False
USE_HALF = True
CLEAR_CUDA_EACH_IMAGE = True

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total GPU memory: {total_mem_gb:.2f} GiB")

print("TILE_SIZE:", TILE_SIZE, "| OVERLAP:", OVERLAP, "| MERGE_IOU:", MERGE_IOU)
print("IMG_SIZE (per tile):", IMG_SIZE)
print("CONF_THRES:", CONF_THRES)
print("IOU_THRES (within-tile):", IOU_THRES)
print("MAX_DET:", MAX_DET)
print("RETINA_MASKS:", RETINA_MASKS)
print("USE_HALF:", USE_HALF)

## 3. Locate the Trained Checkpoint

This cell searches for `.pt` files under `/kaggle/input`.

If more than one checkpoint is found, confirm that the selected path points to the new `img768` model rather than an older `img640` model.

In [12]:
def rank_model_path(path: Path, keywords) -> int:
    """Rank checkpoint candidates using filename/path keywords."""
    text = str(path).lower()
    score = 0

    for keyword in keywords:
        if keyword.lower() in text:
            score += 10

    if path.name.lower() == "best.pt":
        score += 20
    elif "best" in path.name.lower():
        score += 10

    if "last" in path.name.lower():
        score -= 5

    return score


if MANUAL_MODEL_PATH is not None:
    MODEL_PATH = Path(MANUAL_MODEL_PATH)
else:
    model_candidates = sorted(Path("/kaggle/input").rglob("*.pt"))

    if len(model_candidates) == 0:
        raise FileNotFoundError(
            "No .pt checkpoint was found under /kaggle/input. "
            "Please add your trained best.pt as a Kaggle Dataset."
        )

    model_candidates = sorted(
        model_candidates,
        key=lambda p: rank_model_path(p, MODEL_KEYWORDS),
        reverse=True,
    )

    candidate_df = pd.DataFrame({
        "index": list(range(len(model_candidates))),
        "score": [rank_model_path(p, MODEL_KEYWORDS) for p in model_candidates],
        "path": [str(p) for p in model_candidates],
    })

    display(candidate_df)

    MODEL_PATH = model_candidates[MODEL_INDEX]

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"MODEL_PATH does not exist: {MODEL_PATH}")

print("Selected MODEL_PATH:", MODEL_PATH)

,index,score,path
0,0,50,/kaggle/input/models/lingxd/version6-yolov8s-i...


Selected MODEL_PATH: /kaggle/input/models/lingxd/version6-yolov8s-img768-bs16-ep120-fliplr0-best-pt/pytorch/default/1/best.pt


## 4. Load the Official Sample Submission

The sample submission is used as the source of truth for column names and column order.

For this competition, the current sample format is:

```text
id,patient_id,class_id,confidence,poly
```

In [13]:
def find_sample_submission() -> Path:
    """Find the official sample_submission.csv file."""
    preferred_paths = [
        Path("/kaggle/input/competitions/alpha-dent/sample_submission.csv"),
        Path("/kaggle/input/alpha-dent/sample_submission.csv"),
    ]

    for path in preferred_paths:
        if path.exists():
            return path

    candidates = sorted(Path("/kaggle/input").rglob("sample_submission.csv"))

    if len(candidates) == 0:
        raise FileNotFoundError("sample_submission.csv was not found under /kaggle/input.")

    # Prefer paths that look related to AlphaDent.
    candidates = sorted(
        candidates,
        key=lambda p: ("alpha" in str(p).lower(), "dent" in str(p).lower()),
        reverse=True,
    )

    return candidates[0]


SAMPLE_SUBMISSION_PATH = find_sample_submission()
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Selected SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print("Sample submission columns:", sample_submission.columns.tolist())
display(sample_submission.head())

expected_columns = sample_submission.columns.tolist()

required_columns = ["id", "patient_id", "class_id", "confidence", "poly"]
missing_required = [c for c in required_columns if c not in expected_columns]

if len(missing_required) > 0:
    raise ValueError(
        f"The sample submission is missing expected columns: {missing_required}. "
        f"Found columns: {expected_columns}"
    )

Selected SAMPLE_SUBMISSION_PATH: /kaggle/input/competitions/alpha-dent/sample_submission.csv
Sample submission columns: ['id', 'patient_id', 'class_id', 'confidence', 'poly']


,id,patient_id,class_id,confidence,poly
0,0,test_000,7,0.974303,0.1 0.9 0.9 0.9 0.9 0.1 0.1 0.1
1,1,test_000,8,0.879053,0.1 0.9 0.9 0.9 0.9 0.1 0.1 0.1
2,2,test_000,4,0.887693,0.1 0.9 0.9 0.9 0.9 0.1 0.1 0.1
3,3,test_001,1,0.826764,0.1 0.9 0.9 0.9 0.9 0.1 0.1 0.1
4,4,test_001,2,0.163748,0.1 0.9 0.9 0.9 0.9 0.1 0.1 0.1


## 5. Locate Test Images

The code first tries to match image filenames against the `patient_id` values in the official sample submission.  
If that fails, it falls back to searching folders whose path contains `test`.

In [14]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

all_image_paths = sorted([
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

sample_patient_ids = sorted(sample_submission["patient_id"].astype(str).unique())

# First choice: match filenames from sample_submission.
image_by_stem = {}
for path in all_image_paths:
    if path.stem not in image_by_stem:
        image_by_stem[path.stem] = path

matched_paths = [image_by_stem[pid] for pid in sample_patient_ids if pid in image_by_stem]
missing_patient_ids = [pid for pid in sample_patient_ids if pid not in image_by_stem]

if len(matched_paths) > 0:
    test_image_paths = sorted(matched_paths, key=lambda p: p.stem)
    print("Using test images matched from sample_submission patient_id values.")
else:
    test_image_paths = sorted([
        p for p in all_image_paths
        if "test" in str(p).lower()
    ], key=lambda p: p.stem)
    print("Using fallback test image search based on paths containing 'test'.")

print("Number of test images found:", len(test_image_paths))

if len(test_image_paths) == 0:
    raise FileNotFoundError(
        "No test images were found. Please check the competition dataset path."
    )

if len(missing_patient_ids) > 0 and len(matched_paths) > 0:
    print(f"Warning: {len(missing_patient_ids)} patient_id values from sample_submission were not matched to image files.")
    print("First few missing patient IDs:", missing_patient_ids[:10])

print("First few test images:")
for path in test_image_paths[:10]:
    print(" -", path)

Using test images matched from sample_submission patient_id values.
Number of test images found: 135
First few test images:
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_000.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_001.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_002.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_003.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_004.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_005.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_006.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_007.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_008.jpg
 - /kaggle/input/competitions/alpha-dent/AlphaDent/images/test/test_009.jpg


## 6. Helper Functions

`result.masks.xyn` from Ultralytics gives normalized polygons in `[0, 1]` **relative to the
tile** that was predicted. The tiling helpers below (an inline copy of `tools/tile_yolo_seg.py`,
kept in sync) slice each image, map each tile-normalized polygon back to **full-image**
coordinates (`untile_polygon`), and merge duplicate detections from overlapping tiles
(`merge_detections`).

The submission requires each polygon as a full-image-normalized string:

```text
x1 y1 x2 y2 ... xN yN
```</cell id="82b96db8-527f-437f-979d-d1a42e73a031">

In [ ]:
def release_memory():
    """Release Python and CUDA memory after each image."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def polygon_to_string(polygon: np.ndarray) -> str:
    """
    Convert a normalized polygon array with shape [N, 2]
    into a YOLO polygon string:

    x1 y1 x2 y2 ... xN yN
    """
    polygon = np.asarray(polygon, dtype=np.float32)

    # Clip small numerical overflow to the valid range.
    polygon = np.clip(polygon, 0.0, 1.0)

    # Flatten [N, 2] into [x1, y1, x2, y2, ...].
    values = polygon.reshape(-1)

    return " ".join(f"{v:.6f}" for v in values)


def is_valid_polygon(polygon: np.ndarray) -> bool:
    """Return True if the polygon has at least 3 points."""
    if polygon is None:
        return False
    polygon = np.asarray(polygon)
    return polygon.ndim == 2 and polygon.shape[0] >= 3 and polygon.shape[1] == 2


def validate_poly_string(poly: str) -> bool:
    """Validate one polygon string for quick sanity checks."""
    try:
        values = [float(x) for x in str(poly).split()]
    except Exception:
        return False

    if len(values) < 6:
        return False
    if len(values) % 2 != 0:
        return False
    if min(values) < 0.0 or max(values) > 1.0:
        return False

    return True


# ============================================================
# Tiling helpers (inline copy of tools/tile_yolo_seg.py, kept in sync).
# These MUST match the geometry used to build the training tiles in src/01.
# ============================================================

def _starts(length, tile, step):
    if length <= tile:
        return [0]
    s = list(range(0, length - tile + 1, step))
    if s[-1] != length - tile:
        s.append(length - tile)
    return s


def compute_tiles(w, h, tile_size, overlap):
    """Pixel tile boxes (x0, y0, x1, y1) covering the whole image, with edge tiles flush."""
    step = max(1, int(round(tile_size * (1.0 - overlap))))
    boxes = []
    for y0 in _starts(h, tile_size, step):
        for x0 in _starts(w, tile_size, step):
            boxes.append((x0, y0, min(x0 + tile_size, w), min(y0 + tile_size, h)))
    return boxes


def untile_polygon(poly_norm_tile, box, img_w, img_h):
    """Map a tile-normalized polygon back to a full-image-normalized polygon."""
    x0, y0, x1, y1 = box
    tw, th = x1 - x0, y1 - y0
    poly = np.asarray(poly_norm_tile, dtype=np.float64).reshape(-1, 2).copy()
    poly[:, 0] = (poly[:, 0] * tw + x0) / img_w
    poly[:, 1] = (poly[:, 1] * th + y0) / img_h
    return np.clip(poly, 0.0, 1.0)


def _poly_bbox(poly_norm):
    p = np.asarray(poly_norm, dtype=np.float64).reshape(-1, 2)
    return p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max()


def _bbox_iou(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax1 - ax0) * max(0.0, ay1 - ay0)
    area_b = max(0.0, bx1 - bx0) * max(0.0, by1 - by0)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def merge_detections(detections, iou_thres=MERGE_IOU):
    """
    Class-wise greedy NMS over detections from all tiles of one image, de-duplicating
    the same object seen in overlapping tiles. Keeps the highest-confidence copy.

    `detections`: list of dicts with keys class_id:int, confidence:float,
    poly:np.ndarray[N,2] (full-image normalized).
    """
    kept = []
    for det in sorted(detections, key=lambda d: d["confidence"], reverse=True):
        box = _poly_bbox(det["poly"])
        duplicate = False
        for k in kept:
            if k["class_id"] != det["class_id"]:
                continue
            if _bbox_iou(box, _poly_bbox(k["poly"])) > iou_thres:
                duplicate = True
                break
        if not duplicate:
            kept.append(det)
    return kept

## 7. Load Model

The model is loaded from the selected `best.pt` checkpoint.

If you see a CUDA out-of-memory error here, restart the Kaggle session and run the notebook from the beginning.

In [16]:
release_memory()

model = YOLO(str(MODEL_PATH))

print("Model loaded successfully.")
print("Model path:", MODEL_PATH)

Model loaded successfully.
Model path: /kaggle/input/models/lingxd/version6-yolov8s-img768-bs16-ep120-fliplr0-best-pt/pytorch/default/1/best.pt


## 8. Run Tiled Inference

For each test image:

1. read the image and compute its tiles with `compute_tiles` (same geometry as training),
2. run YOLO on each tile crop at `imgsz=TILE_SIZE`,
3. map each predicted polygon back to full-image coordinates with `untile_polygon`,
4. collect all detections across tiles, then merge duplicates from overlapping tiles with
   class-wise NMS (`merge_detections`).

Memory-saving choices: `retina_masks=False`, `half=True` on GPU, one tile at a time,
explicit memory cleanup after each image.</cell id="478f789e-206e-4517-a006-61fb6f2f7cb2">

In [ ]:
def predict_one_tile(tile_img, imgsz, max_det):
    """
    Run YOLO segmentation on a single tile (a BGR numpy crop) and return a list of
    (class_id, confidence, poly_tile_norm[N,2]) for that tile. Polygons are normalized
    to the tile frame (Ultralytics masks.xyn).
    """
    out = []
    with torch.inference_mode():
        results_stream = model.predict(
            source=tile_img,
            imgsz=imgsz,
            conf=CONF_THRES,
            iou=IOU_THRES,
            max_det=max_det,
            device=DEVICE,
            task="segment",
            retina_masks=RETINA_MASKS,
            half=USE_HALF if DEVICE != "cpu" else False,
            stream=True,
            verbose=False,
            save=False,
            save_txt=False,
        )
        for result in results_stream:
            if result.masks is None or result.boxes is None:
                del result
                continue
            polygons = result.masks.xyn
            class_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
            confidences = result.boxes.conf.detach().cpu().numpy().astype(float)
            for class_id, confidence, polygon in zip(class_ids, confidences, polygons):
                if not is_valid_polygon(polygon):
                    continue
                out.append((int(class_id), float(confidence), np.asarray(polygon, dtype=np.float64)))
            del result
    return out


def predict_one_image(image_path: Path, imgsz: int, max_det: int):
    """
    Tiled inference for one full image. Slices the image, predicts each tile, maps every
    polygon back to full-image coordinates, merges overlapping detections, and returns
    submission rows (without the row-level `id`).
    """
    image_path = Path(image_path)
    patient_id = image_path.stem

    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        print(f"WARNING: could not read {image_path.name}; skipping.")
        return []
    h, w = img.shape[:2]

    detections = []
    for (x0, y0, x1, y1) in compute_tiles(w, h, TILE_SIZE, OVERLAP):
        crop = img[y0:y1, x0:x1]
        for class_id, confidence, poly_tile in predict_one_tile(crop, imgsz, max_det):
            poly_full = untile_polygon(poly_tile, (x0, y0, x1, y1), w, h)
            if not is_valid_polygon(poly_full):
                continue
            detections.append({
                "class_id": class_id,
                "confidence": confidence,
                "poly": poly_full,
            })

    merged = merge_detections(detections, iou_thres=MERGE_IOU)

    image_rows = []
    for det in merged:
        image_rows.append({
            "patient_id": patient_id,
            "class_id": int(det["class_id"]),
            "confidence": float(det["confidence"]),
            "poly": polygon_to_string(det["poly"]),
        })
    return image_rows


rows = []

for image_path in tqdm(test_image_paths, desc="Running tiled inference"):
    image_rows = predict_one_image(
        image_path=image_path,
        imgsz=IMG_SIZE,
        max_det=MAX_DET,
    )
    rows.extend(image_rows)

    if CLEAR_CUDA_EACH_IMAGE:
        release_memory()

print("Number of prediction rows (after cross-tile merge):", len(rows))

## 9. Build the Submission DataFrame

The final file must follow the official sample submission column order:

```text
id,patient_id,class_id,confidence,poly
```

Here, `id` is a row-level prediction index, not the image ID.

In [18]:
# Build the submission body first.
submission_df = pd.DataFrame(
    rows,
    columns=["patient_id", "class_id", "confidence", "poly"]
)

# Add row-level prediction ID.
submission_df.insert(0, "id", np.arange(len(submission_df), dtype=int))

# Reorder columns exactly like the official sample submission.
submission_df = submission_df[expected_columns]

# Enforce stable data types.
submission_df["id"] = submission_df["id"].astype(int)
submission_df["patient_id"] = submission_df["patient_id"].astype(str)
submission_df["class_id"] = submission_df["class_id"].astype(int)
submission_df["confidence"] = submission_df["confidence"].astype(float)
submission_df["poly"] = submission_df["poly"].astype(str)

print("Final submission shape:", submission_df.shape)
print("Final submission columns:", submission_df.columns.tolist())
display(submission_df.head())

Final submission shape: (3112, 5)
Final submission columns: ['id', 'patient_id', 'class_id', 'confidence', 'poly']


,id,patient_id,class_id,confidence,poly
0,0,test_000,0,0.893555,0.432292 0.347656 0.432292 0.355469 0.428385 0...
1,1,test_000,2,0.766113,0.252604 0.501953 0.256510 0.501953 0.252604 0...
2,2,test_000,2,0.586914,0.752604 0.464844 0.752604 0.470703 0.747396 0...
3,3,test_000,7,0.586914,0.533854 0.605469 0.533854 0.625000 0.537760 0...
4,4,test_000,7,0.541016,0.585937 0.597656 0.585937 0.623047 0.591146 0...


## 10. Sanity Checks

These checks do not calculate the competition score.  
They only verify that the CSV structure is likely compatible with the Kaggle submission checker.

In [19]:
print("Expected columns:", expected_columns)
print("Actual columns:  ", submission_df.columns.tolist())

assert submission_df.columns.tolist() == expected_columns, "Column order does not match sample_submission.csv."

if len(submission_df) == 0:
    print("Warning: submission_df has zero prediction rows. The file will contain only the header.")
else:
    assert submission_df["class_id"].between(0, 8).all(), "class_id must be between 0 and 8."
    assert submission_df["confidence"].between(0, 1).all(), "confidence must be between 0 and 1."

    poly_valid_flags = submission_df["poly"].head(1000).apply(validate_poly_string)
    valid_rate = poly_valid_flags.mean()

    print(f"Polygon validity rate on first {len(poly_valid_flags)} rows: {valid_rate:.4f}")

    if valid_rate < 1.0:
        invalid_examples = submission_df.loc[~poly_valid_flags].head()
        print("Invalid polygon examples:")
        display(invalid_examples)

    assert valid_rate > 0.99, "Some polygon strings look invalid."

print("Sanity checks completed.")

Expected columns: ['id', 'patient_id', 'class_id', 'confidence', 'poly']
Actual columns:   ['id', 'patient_id', 'class_id', 'confidence', 'poly']
Polygon validity rate on first 1000 rows: 1.0000
Sanity checks completed.


## 11. Save `submission.csv`

After this cell runs, submit:

```text
/kaggle/working/submission.csv
```

In [20]:
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved to: {SUBMISSION_PATH}")

# Read it back to confirm what was actually written.
saved_df = pd.read_csv(SUBMISSION_PATH)

print("Saved submission shape:", saved_df.shape)
print("Saved submission columns:", saved_df.columns.tolist())
display(saved_df.head())

Submission saved to: /kaggle/working/submission.csv
Saved submission shape: (3112, 5)
Saved submission columns: ['id', 'patient_id', 'class_id', 'confidence', 'poly']


,id,patient_id,class_id,confidence,poly
0,0,test_000,0,0.893555,0.432292 0.347656 0.432292 0.355469 0.428385 0...
1,1,test_000,2,0.766113,0.252604 0.501953 0.256510 0.501953 0.252604 0...
2,2,test_000,2,0.586914,0.752604 0.464844 0.752604 0.470703 0.747396 0...
3,3,test_000,7,0.586914,0.533854 0.605469 0.533854 0.625000 0.537760 0...
4,4,test_000,7,0.541016,0.585937 0.597656 0.585937 0.623047 0.591146 0...


## 12. Optional: Inspect Prediction Counts

This is useful for checking whether the confidence threshold is too high or too low.

If the number of rows is extremely small, consider lowering `CONF_THRES`.  
If the number of rows is extremely large, consider increasing `CONF_THRES` or lowering `MAX_DET`.

In [21]:
if len(submission_df) > 0:
    print("Predictions per class:")
    display(submission_df["class_id"].value_counts().sort_index().rename("count").to_frame())

    print("Predictions per image summary:")
    counts_per_image = submission_df.groupby("patient_id").size()
    display(counts_per_image.describe().to_frame(name="predictions_per_image"))

    print("Images with the most predictions:")
    display(counts_per_image.sort_values(ascending=False).head(10).to_frame("num_predictions"))
else:
    print("No predictions were generated.")

Predictions per class:


,count
class_id,
0,1036
1,627
2,183
3,247
4,386
5,197
6,4
7,421
8,11


Predictions per image summary:


,predictions_per_image
count,135.000000
mean,23.051852
std,9.458723
min,4.000000
25%,17.000000
50%,22.000000
75%,28.000000
max,49.000000


Images with the most predictions:


,num_predictions
patient_id,
test_085,49
test_106,48
test_130,47
test_042,46
test_018,44
test_114,41
test_010,41
test_007,41
test_110,40
